<a href="https://colab.research.google.com/github/Yuliia-Safonova/DTA-2026/blob/main/ML/logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [17]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [18]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [19]:
# TODO: виведи баланс класів
print(clients['upgraded'].value_counts(normalize=True).round(3))

# Списки ознак
num_features = ['age', 'tenure', 'usage', 'support']
cat_features = ['plan', 'region']

print('\nЧислові:', num_features)
print('Категорійні:', cat_features)

upgraded
0    0.516
1    0.484
Name: proportion, dtype: float64

Числові: ['age', 'tenure', 'usage', 'support']
Категорійні: ['plan', 'region']


✍️ Випиши списки стовпців (знадобляться далі):
> числові: 'age', 'tenure', 'usage', 'support'  

> категорійні: 'plan', 'region'

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [20]:
from sklearn.model_selection import train_test_split

# TODO: створи X, y та зроби поділ
X = clients[['age', 'tenure', 'usage', 'support', 'plan', 'region']]
y = clients['upgraded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (720, 6)
X_test: (180, 6)
y_train: (720,)
y_test: (180,)


### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: задай num_cols, cat_cols і збери preprocess

num_cols = ['age', 'tenure', 'usage', 'support']
cat_cols = ['plan', 'region']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TODO: збери pipe

pipe = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'tenure', 'usage',
                                                   'support']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plan', 'region'])])),
                ('model', LogisticRegression(max_iter=1000))])


### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [23]:
# TODO: навчи pipe і виведи accuracy

pipe.fit(X_train, y_train)
print('Accuracy на тесті:', round(pipe.score(X_test, y_test), 3))

Accuracy на тесті: 0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [24]:
from sklearn.metrics import confusion_matrix, classification_report

# TODO: передбач, виведи матрицю плутанини та звіт

y_pred = pipe.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# TODO: передбач класи на тесті
y_pred = pipe.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Accuracy = {acc:.2%}\n')

cm = confusion_matrix(y_test, y_pred)

df_cm = pd.DataFrame(
    cm,
    index=['Насправді НЕ ПЕРЕЙШОВ', 'Насправді ПЕРЕЙШОВ'],
    columns=['Передбачено НЕ ПЕРЕЙШОВ', 'Передбачено ПЕРЕЙШОВ']
)

print('Матриця плутанини:')
display(df_cm)

print('\nДетальний звіт по метриках:')
print(
    classification_report(
        y_test,
        y_pred,
        target_names=['НЕ ПЕРЕЙШОВ', 'ПЕРЕЙШОВ']
    )
)

Accuracy = 84.44%

Матриця плутанини:


,Передбачено НЕ ПЕРЕЙШОВ,Передбачено ПЕРЕЙШОВ
Насправді НЕ ПЕРЕЙШОВ,80,13
Насправді ПЕРЕЙШОВ,15,72



Детальний звіт по метриках:
              precision    recall  f1-score   support

 НЕ ПЕРЕЙШОВ       0.84      0.86      0.85        93
    ПЕРЕЙШОВ       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [25]:
from sklearn.metrics import roc_auc_score

# TODO: дістань proba та порахуй ROC-AUC
proba = pipe.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, proba)
print(f'ROC-AUC = {roc_auc:.3f}')
print('\nПриклад ймовірностей перших 5 клієнтів:', proba[:5].round(2))

pd.DataFrame({
    'predict (результат)': pipe.predict(X_test)[:5],
    'predict_proba (ймовірність)': pipe.predict_proba(X_test)[:, 1][:5]
})

ROC-AUC = 0.927

Приклад ймовірностей перших 5 клієнтів: [0.07 0.15 0.   0.97 0.19]


,predict (результат),predict_proba (ймовірність)
0,0,0.067650
1,0,0.151676
2,0,0.001724
3,1,0.968892
4,0,0.187364


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [26]:
# TODO: побудуй таблицю "ознака — коефіцієнт", відсортовану за |коеф.|

feat_names = pipe.named_steps['prep'].get_feature_names_out()
coefs = pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'features': feat_names,
    'coef': coefs
}).sort_values('coef', key=abs, ascending=False)

display(coef_df)

,features,coef
2,num__usage,2.027135
1,num__tenure,1.648386
6,cat__plan_сімейний,1.411953
4,cat__plan_базовий,-1.339728
3,num__support,-0.835947
7,cat__region_захід,0.537637
0,num__age,-0.406875
10,cat__region_схід,-0.340620
8,cat__region_південь,-0.189350
9,cat__region_північ,0.136239


✍️ **Відповідь словами:**
> Найсильніше підвищує шанс переходу на преміум активність використання сервісу (usage), а найбільше знижує — базовий тариф (plan = базовий).  

Загальний висновок:  
Найважливішими факторами зростання ймовірності переходу на преміум є висока активність користування сервісом (usage), довший термін користування сервісом (tenure) та сімейний тариф. Найсильніше знижують імовірність переходу базовий тариф, велика кількість звернень у підтримку та вищий вік клієнта.

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [27]:
# TODO: створи new_client, виведи рішення та ймовірність переходу

new_client = pd.DataFrame([{
    'age': 30,
    'tenure': 24,
    'usage': 120,
    'support': 1,
    'plan': 'сімейний',
    'region': 'захід'
}])

display(new_client)

proba = pipe.predict_proba(new_client)[0, 1]
pred = pipe.predict(new_client)[0]

print(f'\nЙмовірність переходу: {proba:.2%}')
print('Клієнт -', 'ПЕРЕЙДЕ' if pred else 'НЕ ПЕРЕЙДЕ')

,age,tenure,usage,support,plan,region
0,30,24,120,1,сімейний,захід



Ймовірність переходу: 98.89%
Клієнт - ПЕРЕЙДЕ


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [28]:
from sklearn.model_selection import cross_val_score

# TODO: крос-валідація, виведи середнє ± розкид

cv = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')

print(f'CV ROC-AUC: {np.round(cv, 3)} | середнє: {cv.mean():.3f} ± {cv.std():.3f}')

CV ROC-AUC: [0.934 0.931 0.94  0.891 0.921] | середнє: 0.923 ± 0.018


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [29]:
# Місце для бонусних експериментів

# Конвеєр без StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Числові без масштабування
preprocess_no_scale = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

pipe_no_scale = Pipeline([
    ('prep', preprocess_no_scale),
    ('model', LogisticRegression(max_iter=1000))
])

cv_no_scale = cross_val_score(
    pipe_no_scale,
    X, y, cv=5,
    scoring='roc_auc'
)

print(f'CV ROC-AUC без масштабування: {np.round(cv_no_scale, 3)}')
print(f'Середнє: {cv_no_scale.mean():.3f} ± {cv_no_scale.std():.3f}')

CV ROC-AUC без масштабування: [0.935 0.931 0.94  0.891 0.921]
Середнє: 0.923 ± 0.017


Порівнюємо середнє ROC-AUC з попереднім результатом. ROC-AUC змінився незначно.
Масштабування ознак не призвело до суттєвого покращення ROC-AUC. Обидва підходи дають приблизно однакову якість (≈0.923), що свідчить про стабільність моделі відносно масштабів ознак.

In [30]:
# Балансування класів
pipe_balanced = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipe_balanced.fit(X_train, y_train)

y_pred_balanced = pipe_balanced.predict(X_test)

print(classification_report(
    y_test, y_pred_balanced,
    target_names=['НЕ ПЕРЕЙШОВ', 'ПЕРЕЙШОВ']
))

cm_balanced = confusion_matrix(y_test, y_pred_balanced)

df_cm_balanced = pd.DataFrame(
    cm_balanced,
    index=['Насправді НЕ ПЕРЕЙШОВ', 'Насправді ПЕРЕЙШОВ'],
    columns=['Передбачено НЕ ПЕРЕЙШОВ', 'Передбачено ПЕРЕЙШОВ']
)

display(df_cm_balanced)

              precision    recall  f1-score   support

 НЕ ПЕРЕЙШОВ       0.84      0.86      0.85        93
    ПЕРЕЙШОВ       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



,Передбачено НЕ ПЕРЕЙШОВ,Передбачено ПЕРЕЙШОВ
Насправді НЕ ПЕРЕЙШОВ,80,13
Насправді ПЕРЕЙШОВ,15,72


Модель демонструє збалансовану якість класифікації (≈ 0.84) для обох класів. Вона досить добре виявляє клієнтів, які перейдуть на преміум (recall = 0.83), однак частина потенційних преміум-клієнтів залишається не виявленою (false negatives = 15). Це може бути бізнес-ризиком, якщо мета — максимізувати конверсію в преміум.

In [31]:
# Поріг 0.3 замість 0.5

import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    print(f'\nПоріг {thr}')

    cm = confusion_matrix(y_test, pred_thr)
    df_cm = pd.DataFrame(cm,
        index=['Насправді НЕ ПЕРЕЙШОВ', 'Насправді ПЕРЕЙШОВ'],
        columns=['Передбачено НЕ ПЕРЕЙШОВ', 'Передбачено ПЕРЕЙШОВ']
    )

    display(df_cm)

    print(classification_report(
        y_test,
        pred_thr,
        target_names=['НЕ ПЕРЕЙШОВ', 'ПЕРЕЙШОВ']
    ))


Поріг 0.5


,Передбачено НЕ ПЕРЕЙШОВ,Передбачено ПЕРЕЙШОВ
Насправді НЕ ПЕРЕЙШОВ,80,13
Насправді ПЕРЕЙШОВ,15,72


              precision    recall  f1-score   support

 НЕ ПЕРЕЙШОВ       0.84      0.86      0.85        93
    ПЕРЕЙШОВ       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180


Поріг 0.3


,Передбачено НЕ ПЕРЕЙШОВ,Передбачено ПЕРЕЙШОВ
Насправді НЕ ПЕРЕЙШОВ,71,22
Насправді ПЕРЕЙШОВ,8,79


              precision    recall  f1-score   support

 НЕ ПЕРЕЙШОВ       0.90      0.76      0.83        93
    ПЕРЕЙШОВ       0.78      0.91      0.84        87

    accuracy                           0.83       180
   macro avg       0.84      0.84      0.83       180
weighted avg       0.84      0.83      0.83       180



Зниження порогу з 0.5 до 0.3 призвело до суттєвого збільшення recall класу «ПЕРЕЙШОВ» (0.83 → 0.91), тобто модель почала знаходити більше потенційних premium-клієнтів. Водночас зменшилась precision (0.85 → 0.78), що означає збільшення кількості хибнопозитивних прогнозів. Accuracy майже не змінилася, тому зміна порогу впливає не на загальну якість, а на баланс між типами помилок.

---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?
2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?
3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?
4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?
5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.